# Chapter 7 Solutions

This notebook contains solutions to the exercises in Chapter 7 of *Learn Calculus with Python* by Nick McIntyre. Licensed under <a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">CC BY-NC-SA 4.0</a><img src="https://mirrors.creativecommons.org/presskit/icons/cc.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;"><img src="https://mirrors.creativecommons.org/presskit/icons/by.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;"><img src="https://mirrors.creativecommons.org/presskit/icons/nc.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;"><img src="https://mirrors.creativecommons.org/presskit/icons/sa.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;">

In [ ]:
import math
import numpy as np
from ipycc.turtle import Turtle, tracer
import calcplotlib as plt

## Linear Dynamical Systems

### 7-1

In [ ]:
# Show the plot
plt.show()

def frida_diego(state):
    f, d = state
    df_dt = 0.5 * d
    dd_dt = -f
    return np.array([df_dt, dd_dt])

# Prepare to draw
plt.clear()
plt.draw_grid()
plt.draw_axes()
plt.draw_xlabel("F")
plt.draw_ylabel("D")
plt.draw_title("Frida-Diego")

# Compute the trajectory
start = np.array([40, 30])
t_f = 10
trajectory = plt.integrate_field(frida_diego, start, t_f)

# Draw the field
plt.draw_field(frida_diego)

# Draw the trajectory
plt.draw_trajectory_2D(trajectory)

### 7-2

We'll model market goers (demand) and farmers (supply) using differential equations of price as a function of quantity $ \frac{dP_D}{dQ} $ and $ \frac{dP_S}{dQ} $.

Let's say the price consumers are willing to pay increases when their peers demand more and decreases when sellers raise their prices:

$ \frac{dP_D}{dQ} = a P_D $

Let's say the price suppliers charge less when other sellers raise their prices and charge more when consumers demand more:

$ \frac{dS_S}{dQ} = b P_D - c P_S $

In [ ]:
def farmers_market(state):
    P_D, P_S = state
    
    a = 0.05
    b = 0.1
    c = 0.2

    dP_D = a * P_D
    dP_S = b * P_D - c * P_S

    return np.array([dP_D, dP_S])

In [ ]:
# Show the plot
plt.show()

# Prepare to draw
plt.clear()
plt.draw_grid()
plt.draw_axes()
plt.draw_xlabel("D")
plt.draw_ylabel("S")
plt.draw_title("Farmers Market")

# Draw the field
plt.draw_field(farmers_market)

# Compute the trajectory
start = np.array([10, 15])
t_f = 30
trajectory = plt.integrate_field(farmers_market, start, t_f)

# Draw the trajectory
plt.draw_trajectory_2D(trajectory)

### 7-3

In [ ]:
# Configure the time step
t_f = 1
dt = 0.0001
num_steps = int(t_f / dt)

# Create ndarrays to store concentrations at each time step
O = np.zeros(num_steps)
O2 = np.zeros(num_steps)
O3 = np.zeros(num_steps)
NO = np.zeros(num_steps)
NO2 = np.zeros(num_steps)
RO = np.zeros(num_steps)
RO2 = np.zeros(num_steps)

# Set the initial values (some are constant)
O[0] = 0
O2[:] = 5.34 * 10 ** 18
O3[0] = 0
NO[:] = 10 ** 12
NO2[0] = 10 ** 10
RO[:] = 0
RO2[:] = 10 ** 11

# Set the parameters
ka = 7.7 * 10 ** 12
kb = 8.82 * 10 ** (-3)
kc = 6.59 * 10 ** (-34)

for i in range(1, num_steps):
    # Compute the derivatives
    dNO2_dt = ka * NO[i - 1] * RO2[i - 1]
    dO_dt = kb * NO2[i - 1]
    dO3_dt = kc * O[i - 1] * O2[i - 1]
    # Update the concentrations
    NO2[i] = NO2[i - 1] + dt * dNO2_dt
    O[i] = O[i - 1] + dt * dO_dt
    O3[i] = O3[i - 1] + dt * dO3_dt

print(" Chemical | molec/cm^3 ")
print("-" * 23)
print(" " * 6 + f"NO2 | {NO2[-1]:.2e}")
print(" " * 8 + f"O | {O[-1]:.2e} ")
print(" " * 7 + f"O3 | {O3[-1]:.2e}")

In [ ]:
# Scale the concentrations for plotting
NO2 *= 10 ** (-34)
O *= 10 ** (-32)
O3 *= 10 ** (-17)

In [ ]:
# Show the plot
plt.show()

# Create the trajectory
trajectory = np.array([NO2, O3])

# Prepare to draw
plt.clear()
plt.draw_grid()
plt.draw_axes()
plt.draw_xlabel("NO2")
plt.draw_ylabel("O3")
plt.draw_title("NO2-O3")

# Draw the trajectory
plt.draw_trajectory_2D(trajectory)

In [ ]:
# Show the plot
plt.show()

# Create the trajectory
trajectory = np.array([O, O3])

# Prepare to draw
plt.clear()
plt.draw_grid()
plt.draw_axes()
plt.draw_xlabel("O")
plt.draw_ylabel("O3")
plt.draw_title("O-O3")

# Draw the trajectory
plt.draw_trajectory_2D(trajectory)

### 7-4

Here is the system of coupled linear ordinary differential equations that represents this SIR model:

$ \dot{S} = -0.03S + 0.02I $

$ \dot{I} = 0.03S - 0.02I - 0.08I $

$ \dot{R} = 0.08I $

We can use matrix-vector multiplication to express the same model like so:

$ \begin{bmatrix} \dot{S} \\ \dot{I} \\ \dot{R} \end{bmatrix} = \begin{bmatrix} -0.03 & 0.02 & 0 \\ 0.03 & -0.1 & 0 \\ 0 & 0.08 & 0  \end{bmatrix} \begin{bmatrix} S \\ I \\ R \end{bmatrix} $

In [ ]:
def simulate_virus(initial_state):
    # Display the initial state
    S_0, I_0, R_0 = initial_state
    print("====== Start ======")
    print("   S  |   I   |  R")
    print("-" * 20)
    print(f"{S_0:.3f} | {I_0:.3f} | {R_0:.3f}")
    
    # Set the parameters
    A = np.array([
        [-0.03,  0.02, 0],
        [ 0.03, -0.10, 0],
        [    0,  0.08, 0],
    ])
    
    # Configure the time step
    dt = 1
    t_f = 100
    num_steps = int(t_f / dt)
    
    # Run Euler integration
    state = initial_state
    
    for i in range(1, num_steps):
        change = A @ state
        state += dt * change
    
    # Display the results
    S_f, I_f, R_f = state
    print("====== Stop ======")
    print("   S  |   I   |  R")
    print("-" * 20)
    print(f"{S_f:.3f} | {I_f:.3f} | {R_f:.3f}")

S = 0.9
I = 0.1
R = 0
initial_state = np.array([S, I, R])
simulate_virus(initial_state)

# for i in range(10):
#     S = 1 - i / 10
#     I = 1 - S
#     R = 0
#     initial_state = np.array([S, I, R])
#     simulate_virus(initial_state)
#     print(" ")

The initial percentage of susceptible and infected members of the population has a large impact on recovery. In this linear model, each 10% decrease in the in the initial percentage of susceptible individuals increases the final recovery by 5%.

In [ ]:
# Set the initial values
state = np.array([0.5, 0.5, 0.0])

# Set the parameters
A = np.array([
    [-0.03,  0.02, 0],
    [ 0.03, -0.10, 0],
    [    0,  0.08, 0],
])

# Configure the time step
dt = 1
t_f = 100
num_steps = int(t_f / dt)

# Run Euler integration
for i in range(1, num_steps):
    change = A @ state
    state += dt * change

# Display the results
S_f, I_f, R_f = state
print("   S  |   I   |  R")
print("-" * 20)
print(f"{S_f:.3f} | {I_f:.3f} | {R_f:.3f}")

### 7-5

In [ ]:
# Show the plot
plt.show()

# Prepare to draw
plt.clear()
plt.draw_grid()
plt.draw_axes()
plt.draw_xlabel("pos")
plt.draw_ylabel("vel")
plt.draw_title("Mass-Spring")

# Parameters describing the damped mass-spring system
k = 0.5
m = 1
b = 0.1

def damped_mass_spring(state):
    pos, vel = state
    d_pos = vel
    d_vel = -(b / m) * vel - (k / m) * pos
    return np.array([d_pos, d_vel])

# Set the initial values
initial_state = np.array([0.0, 0.0])

# Compute the trajectory
start = np.array([100, 0])
t_f = 20
trajectory = plt.integrate_field(damped_mass_spring, start, t_f)

# Draw the field
plt.draw_field(damped_mass_spring)

# Draw the trajectory
plt.draw_trajectory_2D(trajectory)

## Nonlinear Dynamical Systems

### 7-6

In [ ]:
def lorenz(state):
    x, y, z = state
    sigma = 30
    rho = 84
    beta = 8
    dx_dt = sigma * (y - x)
    dy_dt = x * (rho - z) - y
    dz_dt = x * y - beta * z
    return np.array([dx_dt, dy_dt, dz_dt])

In [ ]:
# Show the plot
plt.show()

# Prepare to draw
plt.clear()
plt.hide_field()

# Transform the points for plotting
S = plt.scale_3D(1, 1, 1)
R = plt.I4 # No rotation
T = plt.translate_3D(0, 0, 0)
A = S @ R @ T

# Draw the axes
plt.draw_axes_3D(A)

# Draw the trajectory in x-y space
t_f = 20
start = np.array([1, 0, 0])
trajectory = plt.integrate_field(lorenz, start, t_f, 3)
plt.draw_trajectory_3D(trajectory, A)

The state of the Lorenz system transitions between two general regions in $ x-y $ space, which appear as quadrants 1 (both variables positive) and 3 (both variables negative). The trajectory may complete a few elliptical "orbits" in a quadrant before transitioning back to the other. There seems to be more variation along the $ y $-axis than the $ x $-axis.

In [ ]:
# Show the plot
plt.show()

# Prepare to draw
plt.clear()

# Transform the points for plotting
S = plt.scale_3D(1, 1, 1)
R = plt.rotate_z(np.pi / 2) @ plt.rotate_y(np.pi / 2)
T = plt.translate_3D(0, 0, 0)
A = S @ R @ T

# Draw the axes
plt.draw_axes_3D(A)

# Draw the trajectory in x-y space
t_f = 20
start = np.array([1, 0, 0])
trajectory = plt.integrate_field(lorenz, start, t_f, 3)
plt.draw_trajectory_3D(trajectory, A)

The state of the Lorenz system transitions between two general regions in $ y-z $ space, which appear as quadrants 1 (both variables positive) and 2 ($ y $ negative and $ z $ positive). The trajectory may complete a few nearly-circular "orbits" in a quadrant before transitioning back to the other. There seems to be almost equal variation along the $ y $-axis and the $ z $-axis.

### 7-7

- $ \alpha $ corresponds to the birth rate of prey
- $ \beta $ corresponds to the death rate of prey
- $ \delta $ corresponds to the birth rate of predators
- $ \gamma $ corresponds to the death rate of predators

### 7-8

In [ ]:
def predator_prey(state):
    x, y = state
    
    alpha = 0.5
    beta = 0.01
    gamma = 0.2
    delta = 0.005

    dx = alpha * x - beta * x * y
    dy = -gamma * y + delta * x * y

    return np.array([dx, dy])

In [ ]:
# Show the plot
plt.show()

# Prepare to draw
plt.clear()
plt.draw_grid()
plt.draw_axes()
plt.draw_xlabel("x")
plt.draw_ylabel("y")
plt.draw_title("Lotka-Volterra")

# Draw the field
plt.draw_field(predator_prey)

# Compute the trajectory
start = np.array([10, 15])
t_f = 30
trajectory = plt.integrate_field(predator_prey, start, t_f)

# Draw the trajectory
plt.draw_trajectory_2D(trajectory)

# Compute the trajectory
start = np.array([10, 10])
t_f = 30
trajectory = plt.integrate_field(predator_prey, start, t_f)

# Draw the trajectory
plt.draw_trajectory_2D(trajectory)

# Compute the trajectory
start = np.array([10, 5])
t_f = 30
trajectory = plt.integrate_field(predator_prey, start, t_f)

# Draw the trajectory
plt.draw_trajectory_2D(trajectory)

Reducing the initial number of predators causes both populations to reach higher maximum values. The shapes of the trajectories are similar.

### 7-9

In [ ]:
def integrate_lotka_volterra(field, initial_state, t_f, num_vars=2, dt=0.001):
    """Integrates a vector field and returns a trajectory"""
    num_steps = int(t_f / dt)
    trajectory = np.zeros([num_vars, num_steps])
    trajectory[:, 0] = initial_state
    
    for i in range(1, num_steps):
        state = trajectory[:, i - 1]
        # Remove 75% of predators
        if i == num_steps // 2:
            state[1] *= 0.25
        change = field(state)
        next_state = state + dt * change
        trajectory[:, i] = next_state
    
    return trajectory

In [ ]:
# Show the plot
plt.show()

# Prepare to draw
plt.clear()
plt.draw_grid()
plt.draw_axes()
plt.draw_xlabel("x")
plt.draw_ylabel("y")
plt.draw_title("Lotka-Volterra")

# Draw the field
plt.draw_field(predator_prey)

# Compute the trajectory
start = np.array([10, 15])
t_f = 30
trajectory = integrate_lotka_volterra(predator_prey, start, t_f)

# Draw the trajectory
plt.draw_trajectory_2D(trajectory)

Removing 75% of predators halfway through the simulation sets the ecosystem on a trajectory with slightly higher maximum populations for both species.

## Agent-Based Modeling

### 7-10

In [ ]:
# Show the plot
plt.show()

# Prepare to draw
plt.clear()
plt.hide_field()

# Create the Turtles
num_food = 50
num_animals = 10
step_size = 5
food = [Turtle() for i in range(num_food)]
animals = [Turtle() for i in range(num_animals)]

In [ ]:
# Show the screen
plt.show()

# Configure the Turtles
tracer(10, 0)
for f in food:
    f.shape("circle")
    f.shapesize(0.2)
    f.color(0.5, 0.5, 0.5)
    f.penup()
    x, y = np.random.uniform(-200, 200, 2)
    f.teleport(x, y)

for a in animals:
    a.shape("turtle")
    a.showturtle()
    a.penup()
    x, y = np.random.uniform(-200, 200, 2)
    a.teleport(x, y)
    angle = np.random.uniform(0, 360)
    a.seth(angle)
tracer(1, 10)

In [ ]:
# Show the plot
plt.show()

# Set the initial values
animal_energy = np.random.uniform(5, 10, num_animals)
food_energy = np.random.uniform(1, 3, num_food)

# Configure the time step
dt = 1
t_f = 100
num_steps = int(t_f / dt)

# Log the number of animals and food at each time step
animal_log = np.zeros(num_steps)
food_log = np.zeros(num_steps)

# Initialize the log values
animal_log[0] = num_animals
food_log[0] = num_food

# Run Euler integration

for i in range(1, num_steps):
    # Eat food
    for a in range(num_animals):
        animal = animals[a]
        if animal_energy[a] > 0:
            # Consume energy
            for f in range(num_food):
                is_available = food_energy[f] > 0
                is_near = animal.distance(food[f]) < 20
                if is_available and is_near:
                    animal_energy[a] += food_energy[f]
                    food_energy[f] = 0
            # Expend energy
            animal_energy[a] -= 0.1
            # Check energy level
            if animal_energy[a] <= 0:
                animal_energy[a] = 0

    # Log the animal and food counts
    animal_log[i] = np.sum(animal_energy > 0)
    food_log[i] = np.sum(food_energy > 0)
    
    # Update the Turtles
    tracer(10, 0)
    for a in range(num_animals):
        if animal_energy[a] > 0:
            animal = animals[a]
            angle = np.random.uniform(-5, 5)
            animal.left(angle)
            animal.forward(1)
            if animal.xcor() > 200:
                animal.setx(-200)
            if animal.xcor() < -200:
                animal.setx(200)
            if animal.ycor() > 200:
                animal.sety(-200)
            if animal.ycor() < -200:
                animal.sety(200)
        else:
            animal.hideturtle()
    for f in range(num_food):
        if food_energy[f] == 0:
            food[f].hideturtle()
    tracer(1, 10)

In [ ]:
# Clear the plot
plt.show()
plt.clear()
tracer(10, 0)
for a in animals:
    a.clear()
    a.hideturtle()
for f in food:
    f.clear()
    f.hideturtle()
tracer(1, 10)

In [ ]:
# Show the plot
plt.show()

# Prepare to draw
plt.clear()
plt.draw_grid()
plt.draw_axes()
plt.draw_xlabel("Food")
plt.draw_ylabel("Animal")
plt.draw_title("Ecosystem")

# Create the trajectory
trajectory = 3 * np.array([food_log, animal_log])

# Draw the trajectory
plt.draw_trajectory_2D(trajectory)

### 7-11

In [ ]:
# Show the plot
plt.show()

# Prepare to draw
plt.clear()

# Create the Turtles
num_birds = 10
num_bugs = 20
num_berries = 30

birds = [Turtle() for i in range(num_birds)]
bugs = [Turtle() for i in range(num_bugs)]
berries = [Turtle() for i in range(num_berries)]

In [ ]:
# Show the plot
plt.show()

# Configure the Turtles
tracer(10, 0)
for b in birds:
    b.penup()
    b.color(0, 0, 0)
    x, y = np.random.uniform(-200, 200, 2)
    b.teleport(x, y)
    angle = np.random.uniform(0, 360)
    b.seth(angle)
    b.showturtle()

for b in bugs:
    b.shape("square")
    b.shapesize(0.3)
    b.penup()
    b.color(0.4, 0.4, 0.4)
    x, y = np.random.uniform(-200, 200, 2)
    b.teleport(x, y)
    angle = np.random.uniform(0, 360)
    b.seth(angle)
    b.showturtle()

for b in berries:
    b.shape("circle")
    b.shapesize(0.2)
    b.penup()
    b.color(0.6, 0.6, 0.6)
    x, y = np.random.uniform(-200, 200, 2)
    b.teleport(x, y)
    b.showturtle()
tracer(1, 10)

# Set the initial values
bird_energy = np.random.uniform(5, 10, num_birds)
bug_energy = np.random.uniform(3, 5, num_bugs)
berry_energy = np.random.uniform(1, 3, num_berries)

# Configure the time step
dt = 1
t_f = 100
num_steps = int(t_f / dt)

# # Create ndarrays for storage
# bird_log = np.zeros(num_steps)
# bug_log = np.zeros(num_steps)
# berry_log = np.zeros(num_steps)

# Run Euler integration
for i in range(1, num_steps):
    # Feed the bugs
    for g in range(num_bugs):
        if bug_energy[g] > 0:
            # Eat nearby berries
            for r in range(num_berries):
                is_available = berry_energy[r] > 0
                is_near = bugs[g].distance(berries[r]) < 20
                if is_available and is_near:
                    bug_energy[g] += berry_energy[r]
                    berry_energy[r] = 0
            # Consume energy
            bug_energy[g] -= 0.01

    # Feed the birds
    for b in range(num_birds):
        if bird_energy[b] > 0:
            # Eat nearby bugs
            for g in range(num_bugs):
                is_available = bug_energy[g] > 0
                is_near = birds[b].distance(bugs[g]) < 20
                if is_available and is_near:
                    bird_energy[b] += bug_energy[g]
                    bug_energy[g] = 0
            # Eat nearby berries
            for r in range(num_berries):
                is_available = berry_energy[r] > 0
                is_near = birds[b].distance(berries[r]) < 20
                if is_available and is_near:
                    bird_energy[b] += berry_energy[r]
                    berry_energy[r] = 0
            # Consume energy
            bird_energy[b] -= 0.05

    # # Update the logs
    # bird_log[i] = np.sum(bird_energy > 0)
    # bug_log[i] = np.sum(bug_energy > 0)
    # berry_log[i] = np.sum(berry_energy > 0)

    tracer(10, 0)
    # Update the berries
    for r in range(num_berries):
        if berry_energy[r] <= 0:
            berries[r].hideturtle()
    # Update the bugs
    for g in range(num_bugs):
        bug = bugs[g]
        if bug_energy[g] > 0:
            angle = np.random.uniform(-5, 5)
            bug.left(angle)
            bug.forward(1)
            if bug.xcor() > 200:
                bug.setx(-200)
            if bug.xcor() < -200:
                bug.setx(200)
            if bug.ycor() > 200:
                bird.sety(-200)
            if bug.ycor() < -200:
                bug.sety(200)
        else:
            bug.hideturtle()
    # Update the birds
    for b in range(num_birds):
        bird = birds[b]
        if bird_energy[b] > 0:
            angle = np.random.uniform(-5, 5)
            bird.left(angle)
            bird.forward(1)
            if bird.xcor() > 200:
                bird.setx(-200)
            if bird.xcor() < -200:
                bird.setx(200)
            if bird.ycor() > 200:
                bird.sety(-200)
            if bird.ycor() < -200:
                bird.sety(200)
        else:
            bird.hideturtle()
    tracer(1, 10)

In [ ]:
# Clear the plot
plt.show()
tracer(10, 0)
for b in birds:
    b.clear()
    b.hideturtle()
for b in bugs:
    b.clear()
    b.hideturtle()
for b in berries:
    b.clear()
    b.hideturtle()
tracer(1, 10)

### 7-12

In [ ]:
# Show the plot
plt.show()

# Create the Turtles
num_people = 100
people = [Turtle() for i in range(num_people)]

In [ ]:
# Show the plot
plt.show()

# Set the initial values
num_susceptible = 95
num_infected = 5
num_recovered = 0
health = np.zeros(num_people)
health[0:num_susceptible] = 1
health[num_susceptible:num_susceptible + num_infected] = 0.95

# Configure the Turtles
tracer(10, 0)
for i in range(num_people):
    p = people[i]
    p.penup()
    p.shape("circle")
    if i < num_susceptible or i >= num_susceptible + num_infected:
        p.color(plt.BLACK)
    if num_susceptible <= i < num_susceptible + num_infected:
        p.color(plt.GRAY60)
    x, y = np.random.uniform(-200, 200, 2)
    p.teleport(x, y)
    angle = np.random.uniform(0, 360)
    p.seth(angle)
    p.showturtle()
tracer(1, 10)

# Configure the time step
dt = 1
t_f = 100
num_steps = int(t_f / dt)

# Run Euler integration
for i in range(1, num_steps):
    for p in range(num_people):
        # Let the infection run its course
        if 0 < health[p] < 1:
            health[p] -= 0.02
        # Check for new infections
        if health[p] == 1: 
            for other in range(num_people):
                is_infected = 0 < health[other] < 1
                is_near = people[p].distance(people[other]) < 20
                is_transmitted = np.random.uniform() < 0.1
                if is_infected and is_near and is_transmitted:
                    health[p] = 0.95
        # Check for recovery
        if health[p] <= 0:
            health[p] = 100

    # Update the Turtles
    tracer(10, 0)
    for p in range(num_people):
        person = people[p]
        angle = np.random.uniform(-5, 5)
        person.left(angle)
        person.forward(1)
        if person.xcor() > 200:
            person.setx(-200)
        if person.xcor() < -200:
            person.setx(200)
        if person.ycor() > 200:
            person.sety(-200)
        if person.ycor() < -200:
            person.sety(200)
        # Susceptible
        if health[p] == 1:
            person.color(plt.BLACK)
        # Infected
        if 0 < health[p] < 1:
            person.color(plt.GRAY60)
        # Recovered
        if health[p] == 100:
            person.color(plt.WHITE)
    tracer(1, 10)

In [ ]:
# Clear the plot
tracer(10, 0)
plt.show()
for p in people:
    p.clear()
    p.hideturtle()
tracer(1, 10)